# Introduction to privJedAI

This notebook will guide you throughout all the possible methods and how to use them by our open-source library privJedAI.

In [1]:
%pip install privjedai


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached numpy-2.2.6-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (62 kB)
  Using cached pandas-2.3.3-cp310-cp310-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (91 kB)
  Using cached scipy-1.15.3-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached ipywidgets-8.1.8-py3-none-any.whl.metadata (2.4 kB)
  Using cached widgetsnbextension-4.0.15-py3-none-any.whl.metadata (1.6 kB)
  Using cached jupyterlab_widgets-3.0.16-py3-none-any.whl.metadata (20 kB)
  Using cached pytz-2026.1.post1-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2025.3-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached click-8.3.1-py3-none-any.whl.metadata (2.6 kB)
  Using cached jsonschema-4.26.0-py3-none-any.whl.metadata (7.6 kB)
  Usi

## Import Dataset Abt Buy Clean

Below we load the two different datasets and their ground truth. privJedAI needs the indices of the pairs for each dataset and not their id. Here we present also how to preprocess a ground truth that contains id1-id2 pairs to index1-index2 pairs.

In [1]:
import pandas as pd

DIR = "D2"
PATH = f"data/{DIR}"
FILE = 'abtclean'
FILE2 = 'buyclean'
attributes = ['name']
SEP = "|"

d1 = pd.read_csv(f"../{PATH}/{FILE}.csv", sep=SEP)
d2 = pd.read_csv(f"../{PATH}/{FILE2}.csv", sep=SEP)

gt = pd.read_csv(f'../{PATH}/gtclean.csv' , sep=SEP)
df_a = d1.reset_index().rename(columns={"index": "index_A"})
df_b = d2.reset_index().rename(columns={"index": "index_B"})

df_a = df_a[["index_A", "id"]]
df_b = df_b[["index_B", "id"]]

gt_index = gt.merge(left_on='D1', right=df_a, right_on='id')

gt_index = gt_index.drop(columns=['id', 'D1'])
gt_index.columns = ['D2', 'D1']

gt_index = gt_index.merge(left_on='D2', right=df_b, right_on='id')
gt_index = gt_index.drop(columns=['id', 'D2'])
gt_index.columns = ['D1', 'D2']
d1 = d1.astype(str)
d2 = d2.astype(str)

d1.head()


,id,name,description,price
0,0,Sony Turntable - PSLX350H,Sony Turntable - PSLX350H/ Belt Drive System/ ...,nan
1,1,Bose Acoustimass 5 Series III Speaker System -...,Bose Acoustimass 5 Series III Speaker System -...,399.0
2,2,Sony Switcher - SBV40S,Sony Switcher - SBV40S/ Eliminates Disconnecti...,49.0
3,3,Sony 5 Disc CD Player - CDPCE375,Sony 5 Disc CD Player- CDPCE375/ 5 Disc Change...,nan
4,4,Bose 27028 161 Bookshelf Pair Speakers In Whit...,Bose 161 Bookshelf Speakers In White - 161WH/ ...,158.0


## Encode data and build bloom filters

Each party agree in an exact configuration and then encode locally their data.
Those encoded data are then shared to a third party to proceed with record linkage.

In [2]:
from privjedai.encoder import BloomFilterConfig, BloomEncodedData, BloomFilter

bloom_filter_configuration = {
    "size" : 512,
    "offset" : 0,
    "num_hashes" : 15,
    "hashing_type": "salted_qgrams",
    "salt" : "",
    "attributes": ['name'],
    "qgrams": 4
}

config = BloomFilterConfig(**bloom_filter_configuration)
bloom_generator = BloomFilter(config)

## The two parties encode their datasets and save them to disk.
## The encoded datasets are then shared with the other party and used for the matching process.
encoded_d1 = bloom_generator.encode(d1)
encoded_d1.to_file(f"dataset_1.pkl")

encoded_d2 = bloom_generator.encode(d2)
encoded_d2.to_file(f"dataset_2.pkl")

Encoding Data with attributes ['name']:   0%|          | 0/1076 [00:00<?, ?it/s]

Encoding Data with attributes ['name']:   0%|          | 0/1064 [00:00<?, ?it/s]

In [3]:
# Third party loads the encoded datasets and performs the matching process.
encoded_data = BloomEncodedData.from_file("dataset_1.pkl", "dataset_2.pkl")


# Ground truth must be explicitly set for the evaluation process. 
# This is done by providing the indices of the matching records in the original datasets.
encoded_data.set_ground_truth(gt_index)

## Blocking with privJedAI

In privJedAI we have 2 different implementations of Hamming Blocking and a FAISS implementation.

### BitBlocker

In [4]:
from privjedai.blocking import BitBlocker

blocker = BitBlocker(psi = 8,
            lmbda = 24,
            seed = 42)

blocks = blocker.build_blocks(encoded_data=encoded_data)

_ = blocker.evaluate(blocks)



***************************************************************************************************************************
                                         Method:  BitBlocker
***************************************************************************************************************************
Method name: BitBlocker
Parameters: 
Runtime: 0.2179 seconds
───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Performance:
	Precision:      0.45% 
	Recall:        85.81%
	F1-score:       0.90%
───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


### LSHBlocker


In [5]:
from privjedai.blocking import LSHBlocker

blocker = LSHBlocker(psi = 8,
            lmbda = 24,
            seed = 42,
            prune_ratio = 0.8)

blocks = blocker.build_blocks(encoded_data=encoded_data)
_ = blocker.evaluate(blocks)

***************************************************************************************************************************
                                         Method:  LSHBlocker
***************************************************************************************************************************
Method name: LSHBlocker
Parameters: 
	psi: 8
	lmbda: 24
	prune_ratio: 0.8
	prune_sample: 1000
	seed: 42
Runtime: 0.2561 seconds
───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Performance:
	Precision:      0.10% 
	Recall:        92.86%
	F1-score:       0.19%
───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


### FAISSBlocking

In [6]:
from privjedai.blocking import FAISSBlocking

blocker = FAISSBlocking(index_type='hnsw')

blocks = blocker.build_blocks(encoded_data=encoded_data, top_k=10)
_ = blocker.evaluate(blocks)

***************************************************************************************************************************
                                         Method:  FAISS Blocking
***************************************************************************************************************************
Method name: FAISS Blocking
Parameters: 
Runtime: 0.0496 seconds
───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Performance:
	Precision:      8.59% 
	Recall:        85.90%
	F1-score:      15.62%
───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


## Meta-blocking Techniques

Those above are all standard blocking techniques. privJedAI also implements meta-blocking methods. It leverages comparison cleaning methods from ER and implements them for bitarrays. A block is a set of adjacent active bits of a bitarray.

In [7]:
from privjedai.comparison_cleaning import CardinalityNodePruning

cc = CardinalityNodePruning(weighting_scheme='CN-CBS')

cc_blocks = cc.process(encoded_data, adjacent_bits=12)

_ = cc.evaluate(cc_blocks)

Total matching pairs: 41694
***************************************************************************************************************************
                                         Method:  Cardinality Node Pruning
***************************************************************************************************************************
Method name: Cardinality Node Pruning
Parameters: 
	Node centric: True
	Weighting scheme: CN-CBS
Runtime: 0.2381 seconds
───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Performance:
	Precision:      1.94% 
	Recall:        76.13%
	F1-score:       3.79%
───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


## Matching

After filtering our datasets we can use a similarity function and match the possible candidate pairs.

In [8]:
from privjedai.matching import Matcher
import numpy as np
matcher = Matcher(batch_size = 10_000,
                  threshold = 0.5,
                  metric='cosine')

matches = matcher.predict(encoded_data=encoded_data, blocks=blocks)

_ = matcher.evaluate(matches)

Predicting batches:   0%|          | 0/2 [00:00<?, ?it/s]

***************************************************************************************************************************
                                         Method:  Matcher
***************************************************************************************************************************
Method name: Matcher
Parameters: 
	batch_size: 10000
	threshold: 0.5
	metric: cosine
	attributes: None
Runtime: 0.0713 seconds
───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Performance:
	Precision:      8.72% 
	Recall:        85.90%
	F1-score:      15.84%
───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


## Clustering

To eliminate possible conflicts on the matching pairs, we provide multiple clustering techniques.

In [9]:
from privjedai.clustering import KiralyMSMApproximateClustering

clusterer = KiralyMSMApproximateClustering()

clusters = clusterer.process(matches, encoded_data=encoded_data, similarity_threshold=0.5)

_ = clusterer.evaluate(clusters)

***************************************************************************************************************************
                                         Method:  Kiraly MSM Approximate Clustering
***************************************************************************************************************************
Method name: Kiraly MSM Approximate Clustering
Parameters: 
	Similarity Threshold: 0.5
Runtime: 0.0156 seconds
───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Performance:
	Precision:     81.85% 
	Recall:        74.15%
	F1-score:      77.81%
───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
